In [ ]:
import sys
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Folder:", Path.cwd().name)

for name in ["numpy", "pandas", "sklearn"]:
    try:
        __import__(name)
        print(name, "- ok")
    except ImportError:
        print(name, "- missing")

Python: 3.13.15
Folder: content
numpy - ok
pandas - ok
sklearn - ok


# Build the dataset

In [ ]:
import csv
from pathlib import Path
import numpy as np

SEED = 42
N_ROWS = 600
DATA = Path("data") / "delivery_times.csv"

def make_delivery_csv(path=DATA):
    rng = np.random.default_rng(SEED)
    distance_km = np.round(rng.uniform(0.5, 12.0, N_ROWS), 2)
    prep_time_min = np.round(rng.uniform(5, 30, N_ROWS), 0)
    traffic_level = rng.integers(1, 4, N_ROWS)
    rain = rng.binomial(1, 0.25, N_ROWS)
    delivery_min = np.round(
        6.0 + 3.1 * distance_km + 0.65 * prep_time_min
        + 4.2 * traffic_level + 5.5 * rain
        + rng.normal(0, 2.5, N_ROWS), 1)

    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["distance_km", "prep_time_min", "traffic_level", "rain", "delivery_min"])
        for i in range(N_ROWS):
            w.writerow([distance_km[i], int(prep_time_min[i]), int(traffic_level[i]), int(rain[i]), delivery_min[i]])
    return path


if not DATA.exists():
    make_delivery_csv()
print("dataset ready:", DATA)


dataset ready: data/delivery_times.csv


# Loads the data, separates features (X) from the target (y), and does an 80/20 train/test split.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

orders = pd.read_csv(DATA)
FEATURES = ["distance_km", "prep_time_min", "traffic_level", "rain"]
X = orders[FEATURES]
y = orders["delivery_min"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(len(X_train), len(X_test))

480 120


# Define a function that trains a model, measures its error on both the training data and the test data, and computes the gap between them.

In [ ]:
from sklearn.metrics import mean_absolute_error

def score_both_ways(model, name):
    model.fit(X_train, y_train)
    train_mae = mean_absolute_error(y_train, model.predict(X_train))
    test_mae = mean_absolute_error(y_test, model.predict(X_test))
    gap = test_mae - train_mae
    print(name, "train", round(train_mae, 2), "test", round(test_mae, 2), "gap", round(gap, 2))
    return {"name": name, "train": train_mae, "test": test_mae, "gap": gap}

# Model 1: LinearRegression

In [ ]:
from sklearn.linear_model import LinearRegression

linear = score_both_ways(LinearRegression(), "LinearRegression")

LinearRegression train 2.04 test 1.92 gap -0.11


# Model 2: decision tree, no limit

In [ ]:
from sklearn.tree import DecisionTreeRegressor

wild_tree = score_both_ways(DecisionTreeRegressor(random_state=42), "DecisionTree (no limit)")

DecisionTree (no limit) train 0.0 test 3.43 gap 3.43


# Model 3: shallow tree (depth 4)

In [ ]:
small_tree = score_both_ways(DecisionTreeRegressor(max_depth=4, random_state=42), "DecisionTree (depth 4)")

DecisionTree (depth 4) train 3.66 test 4.23 gap 0.57


# Model 4: RandomForest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest = score_both_ways(RandomForestRegressor(n_estimators=50, random_state=42), "RandomForest (50 trees)")

RandomForest (50 trees) train 1.0 test 2.39 gap 1.4


In [ ]:
results = pd.DataFrame([linear, wild_tree, small_tree, forest]).round(2).sort_values("test")
print(results.to_string(index=False))


                   name  train  test   gap
       LinearRegression   2.04  1.92 -0.11
RandomForest (50 trees)   1.00  2.39  1.40
DecisionTree (no limit)   0.00  3.43  3.43
 DecisionTree (depth 4)   3.66  4.23  0.57


# Cross-validation - Trains and tests each model 5 times on 5 different data splits and averages the error

In [ ]:
from sklearn.model_selection import cross_val_score

def cross_validate(model, name):
    scores = -cross_val_score(model, X, y, cv=5, scoring="neg_mean_absolute_error")
    print(name, "MAE", round(scores.mean(), 2))
    return scores.mean()

cv_linear = cross_validate(LinearRegression(), "LinearRegression")
cv_tree = cross_validate(DecisionTreeRegressor(max_depth=4, random_state=42), "DecisionTree (depth 4)")
cv_forest = cross_validate(RandomForestRegressor(n_estimators=50, random_state=42), "RandomForest (50 trees)")

LinearRegression MAE 2.03
DecisionTree (depth 4) MAE 4.57
RandomForest (50 trees) MAE 2.69


# Tries several tree depths (2, 3, 4, 6, 8, None), scores each with cross-validation, and finds the best one.

In [ ]:
depths = [2, 3, 4, 6, 8, None]
scores_by_depth = {}

for depth in depths:
    tree = DecisionTreeRegressor(max_depth=depth, random_state=42)
    scores = cross_val_score(tree, X, y, cv=5,
                             scoring="neg_mean_absolute_error")
    mae = -scores.mean()
    scores_by_depth[depth] = round(float(mae), 3)

best_depth = min(scores_by_depth, key=scores_by_depth.get)
print(scores_by_depth)
print("best depth:", best_depth)

{2: 5.858, 3: 4.85, 4: 4.573, 6: 3.642, 8: 3.395, None: 3.478}
best depth: 8


# Ranks the three model

In [ ]:
ranking = sorted(
    {"LinearRegression": cv_linear, "DecisionTree(4)": cv_tree, "RandomForest(50)": cv_forest}.items(),
    key=lambda kv: kv[1],
)
for name, mae in ranking:
    print(name, round(mae, 2))

LinearRegression 2.03
RandomForest(50) 2.69
DecisionTree(4) 4.57


# To Do: Train Logistic Regression, Decision Tree, and Random Forest classifiers on the Breast Cancer dataset, compare their train/test accuracy, use cross-validation to find the best tree depth, and report the confusion matrix and classification metrics to identify the best-performing model. https://www.kaggle.com/datasets/yasserh/breast-cancer-dataset

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

# Load the dataset
cancer = load_breast_cancer()
X_cancer, y_cancer = cancer.data, cancer.target

# Split the data into training and testing sets
X_train_cancer, X_test_cancer, y_train_cancer, y_test_cancer = train_test_split(X_cancer, y_cancer, test_size=0.2, random_state=42)

### Classification Model Scoring Function

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

def score_classifier_both_ways(model, name):
    model.fit(X_train_cancer, y_train_cancer)

    y_train_pred = model.predict(X_train_cancer)
    y_test_pred = model.predict(X_test_cancer)

    train_acc = accuracy_score(y_train_cancer, y_train_pred)
    test_acc = accuracy_score(y_test_cancer, y_test_pred)

    print(f"{name} train accuracy: {train_acc:.4f}")
    print(f"{name} test accuracy: {test_acc:.4f}")

    return {"name": name, "train_accuracy": train_acc, "test_accuracy": test_acc, "model": model}


### Model 1: Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

logistic_results = score_classifier_both_ways(LogisticRegression(max_iter=1000, random_state=42), "LogisticRegression")

LogisticRegression train accuracy: 0.9582
LogisticRegression test accuracy: 0.9561


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### Model 2: Decision Tree Classifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier

wild_tree_classifier_results = score_classifier_both_ways(DecisionTreeClassifier(random_state=42), "DecisionTree (no limit)")

DecisionTree (no limit) train accuracy: 1.0000
DecisionTree (no limit) test accuracy: 0.9474


### Model 3: Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest_classifier_results = score_classifier_both_ways(RandomForestClassifier(n_estimators=100, random_state=42), "RandomForest")

RandomForest train accuracy: 1.0000
RandomForest test accuracy: 0.9649


### Comparing Model Accuracies

In [ ]:
import pandas as pd

results_df = pd.DataFrame([logistic_results, wild_tree_classifier_results, forest_classifier_results])
results_df = results_df.drop(columns=['model'])
print(results_df.sort_values(by='test_accuracy', ascending=False).to_string(index=False))

                   name  train_accuracy  test_accuracy
           RandomForest        1.000000       0.964912
     LogisticRegression        0.958242       0.956140
DecisionTree (no limit)        1.000000       0.947368


### Cross-validation for Decision Tree Depth

In [ ]:
from sklearn.model_selection import cross_val_score

depths = [2, 3, 4, 6, 8, 10, None]
scores_by_depth_classifier = {}

for depth in depths:
    tree_classifier = DecisionTreeClassifier(max_depth=depth, random_state=42)
    scores = cross_val_score(tree_classifier, X_cancer, y_cancer, cv=5, scoring="accuracy")
    avg_accuracy = scores.mean()
    scores_by_depth_classifier[depth] = round(float(avg_accuracy), 4)

best_depth_classifier = max(scores_by_depth_classifier, key=scores_by_depth_classifier.get)
print("Cross-validation accuracies by depth:", scores_by_depth_classifier)
print("Best depth for Decision Tree Classifier:", best_depth_classifier)

Cross-validation accuracies by depth: {2: 0.928, 3: 0.9191, 4: 0.9209, 6: 0.9209, 8: 0.9156, 10: 0.9173, None: 0.9173}
Best depth for Decision Tree Classifier: 2


### Evaluate Best Performing Model (based on test accuracy from the initial comparison)

In [ ]:
# Based on the previous comparison, identify the model with the highest test accuracy
best_model_name = results_df.loc[results_df['test_accuracy'].idxmax()]['name']

# Retrain the best model if necessary (or get it from the stored results)
if best_model_name == "LogisticRegression":
    best_classifier = LogisticRegression(max_iter=1000, random_state=42)
elif best_model_name == "DecisionTree (no limit)":
    best_classifier = DecisionTreeClassifier(random_state=42)
elif best_model_name == "RandomForest":
    best_classifier = RandomForestClassifier(n_estimators=100, random_state=42)

best_classifier.fit(X_train_cancer, y_train_cancer)
y_pred_best = best_classifier.predict(X_test_cancer)

print(f"\nConfusion Matrix for {best_model_name}:")
print(confusion_matrix(y_test_cancer, y_pred_best))

print(f"\nClassification Report for {best_model_name}:")
print(classification_report(y_test_cancer, y_pred_best))


Confusion Matrix for RandomForest:
[[40  3]
 [ 1 70]]

Classification Report for RandomForest:
              precision    recall  f1-score   support

           0       0.98      0.93      0.95        43
           1       0.96      0.99      0.97        71

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114

